In [ ]:
"""
LeanIX Data Pipeline - Parameterized Notebook
Fetches LeanIX application data via GraphQL and exports to Databricks + S3

Parameters:
  - catalog: Databricks catalog name
  - schema: Databricks schema name
  - cdh_bucket: S3 CDH bucket path
  - leanix_subdomain: LeanIX subdomain (e.g., 'portfolio')
  - leanix_api_token_scope: Databricks secret scope
  - leanix_api_token_key: Secret key for API token
  - output_volume_path: Volume path for JSON exports
  - environment: Target environment (dev/preprod/prod)
"""

import os
import sys
import requests
import json
import pandas as pd
import re
from datetime import datetime

# ============================================================================
# CONFIGURATION & PARAMETERS
# ============================================================================

# Databricks widget parameters (passed from job)
dbutils.widgets.text("catalog", "leanix_dev", "Catalog")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("cdh_bucket", "s3://demo-data-bucket/", "CDH Bucket")
dbutils.widgets.text("leanix_subdomain", "portfolio", "LeanIX Subdomain")
dbutils.widgets.text("leanix_api_token_scope", "leanix-secrets-dev", "API Token Scope")
dbutils.widgets.text("leanix_api_token_key", "api_token", "API Token Key")
dbutils.widgets.text("output_volume_path", "/Volumes/leanix_dev/default/leanix_exports", "Output Volume Path")
dbutils.widgets.text("environment", "dev", "Environment")

# Retrieve parameter values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
cdh_bucket = dbutils.widgets.get("cdh_bucket")
leanix_subdomain = dbutils.widgets.get("leanix_subdomain")
leanix_api_token_scope = dbutils.widgets.get("leanix_api_token_scope")
leanix_api_token_key = dbutils.widgets.get("leanix_api_token_key")
output_volume_path = dbutils.widgets.get("output_volume_path")
environment = dbutils.widgets.get("environment")

# Create database if not exists
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

print(f"✓ Configuration loaded for environment: {environment}")
print(f"  Catalog: {catalog}.{schema}")
print(f"  CDH Bucket: {cdh_bucket}")

# ============================================================================
# LEANIX API AUTHENTICATION
# ============================================================================

def get_leanix_api_token():
    """Fetch LeanIX API token from Databricks secrets"""
    try:
        token = dbutils.secrets.get(scope=leanix_api_token_scope, key=leanix_api_token_key)
        if token:
            return token
        else:
            raise Exception("Token is empty")
    except Exception as e:
        print(f"⚠️ Warning: Could not read from secrets scope '{leanix_api_token_scope}': {e}")
        print("   Falling back to environment variable LEANIX_API_TOKEN")
        return os.environ.get('LEANIX_API_TOKEN', '')

LEANIX_API_TOKEN = get_leanix_api_token()
LEANIX_GRAPHQL_URL = f'https://{leanix_subdomain}.leanix.net/services/pathfinder/v1/graphql'
LEANIX_OAUTH2_URL = f'https://{leanix_subdomain}.leanix.net/services/mtm/v1/oauth2/token'

def obtain_access_token():
    """Obtain LeanIX Access token using API secret"""
    if not LEANIX_API_TOKEN:
        raise Exception('❌ No valid LeanIX token found')

    response = requests.post(
        LEANIX_OAUTH2_URL,
        auth=("apitoken", LEANIX_API_TOKEN),
        data={"grant_type": "client_credentials"},
    )
    response.raise_for_status()
    return response.json().get('access_token')

print("→ Fetching LeanIX access token...")
access_token = obtain_access_token()
print("✓ LeanIX authentication successful")

# ============================================================================
# LEANIX GRAPHQL QUERY
# ============================================================================

graphql_query = """query allFactSheetsQuery($filter: FilterInput!, $sortings: [Sorting]) {
  allFactSheets(first: 816, filter: $filter, sort: $sortings) {
    totalCount
    pageInfo {
      hasNextPage
      hasPreviousPage
      startCursor
      endCursor
    }
    edges {
      node {
        ... on Application {
          displayName
          name
          description
          category
          completion {
            percentage
          }
          id
          fullName
          type
          status
          level
          createdAt
          updatedAt
          lxState
          alias
          externalId {
            externalId
          }
          LoginURL
          QoS
          lxCatalogStatus
          lifecycle {
            asString
            phases {
              phase
              startDate
              milestoneId
            }
          }
          CAPEX_per_year
          OPEX_per_year
          CAPEX2024
          OPEX2024
          CAPEX2025
          OPEX2025
          CAPEX2026
          OPEX2026
          CurrencyFinancail
          CostCenter
          OPEXReference
          lxTimeClassification
          lxTimeClassificationDescription
          businessCriticality
          functionalSuitability
          businessCriticalityDescription
          functionalSuitabilityDescription
          UserSupportHours
          ApplicationType
          AuthenticationMethod
          OtherAuthenticationMethod
          AssetType
          ApplicationExposure
          ZPAusage
          ZPArestriction
          PartnerAccessZscaler
          AssetPriority
          AdditionalSecurityMeasures
          AutomatedSecurityScans
          lxHostingType
          HostingArea
          CloudServiceProvider
          OtherCloudServiceProvider
          CloudAccount
          Region
          location {
            asString
          }
          ApplicationEnvironments
          ISPDone
          DateOfEvaluation
          CIATConfidentiality
          CIATIntegrity
          CIATAvailability
          CIATTracability
          SAPDone
          SAPDate
          PenetrationTestDone
          DateOfLastPenetrationTest
          PrivacyAssessmentDone
          DateOfPrivacyAssessment
          ProcomValidation
          PersonalDataProcessed
          TypeOfPersonalData
          PrivacySensitivityScore
          PrivacyComplianceLevel
          DRPObligation
          DRPAvailability
          RTO
          RPO
          DRPFearedEvents
          DRPLastTestDate
          DRPNextTestDate
          BackupStrategy
          BackupImplemented
          BackupDeletionProtection
          AccessibilityAuditDone
          LastAccessibilityAuditDate
          DigitalAccessibiltyScore
          DigitalAccessibilityStatement
          DateDigitalAccessibilityStatement
          relApplicationToApplicationPortfolio {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToPredecessor {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToSuccessor {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToStandardSolution {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToRequires {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToRequiredBy {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToParent {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToChild {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relInstantiateApplicationToInstantiatedApplication {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relInstantiatedApplicationToInstantiateApplication {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToInitiative {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToBusinessCapability {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToBusinessContext {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToOrganization {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToDataObject {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relProviderApplicationToInterface {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relConsumerApplicationToInterface {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToProvider {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToITComponent {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          subscriptions {
            edges {
              node {
                id
                user {
                  id
                  firstName
                  lastName
                  email
                  permission {
                    role
                    status
                  }
                }
                type
                roles {
                  id
                  name
                  subscriptionType
                  comment
                }
                createdAt
              }
            }
          }
          tags {
            id
            name
            color
            tagGroup {
              name
              shortName
            }
          }
          rev
          naFields
          permissions {
            create
            read
            update
            delete
          }
        }
      }
    }
  }
}
"""

variables = {
    "filter": {
        "facetFilters": [
            {"facetKey": "FactSheetTypes", "operator": "OR", "keys": ["Application"]},
            {
                "facetKey": "00000000-0000-0000-0000-000000000000",
                "operator": "OR",
                "keys": ["00000000-0000-0000-0000-000000000000"],
            },
        ]
    },
    "sortings": [{"key": "displayName", "order": "asc"}],
}

# ============================================================================
# FETCH DATA FROM LEANIX
# ============================================================================

print("→ Fetching data from LeanIX GraphQL API...")

data = {'query': graphql_query, 'variables': variables}
auth_header = f'Bearer {access_token}'

response = requests.post(
    url=LEANIX_GRAPHQL_URL,
    headers={'Authorization': auth_header},
    data=json.dumps(data),
)
response.raise_for_status()

result = response.json()

# Save raw JSON
os.makedirs(output_volume_path, exist_ok=True)
output_json_file = f"{output_volume_path}/leanix_raw_result_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(output_json_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

edges = result.get("data", {}).get("allFactSheets", {}).get("edges", [])
print(f"✓ Fetched {len(edges)} applications from LeanIX")
print(f"✓ Raw JSON saved to: {output_json_file}")

# ============================================================================
# DATA TRANSFORMATION (Rest of original notebook logic)
# ============================================================================

ROLE_BUSINESS_OWNER = ["Business Application Owner"]
ROLE_IT_OWNER       = ["IT Application Owner"]
ROLE_TECHNICAL      = ["Architect", "IT Contact"]
ROLE_DPO            = ["Data Privacy Manager"]
ROLE_CISO           = ["CISO"]

def g(d, *p, default=None):
    cur = d
    for k in p:
        if not isinstance(cur, dict): return default
        cur = cur.get(k)
        if cur is None: return default
    return cur

def fmt(v):
    if v is None: return None
    if isinstance(v, bool): return "Oui" if v else "Non"
    if isinstance(v, list): return ", ".join(str(x) for x in v if x not in (None, "")) or None
    return str(v).strip() or None

def field(k): return lambda n: fmt(n.get(k))
def combo(*ks, sep=" — "): return lambda n: sep.join(x for x in (fmt(n.get(k)) for k in ks) if x) or None
def rel_names(n, rel): return [g(e,"node","factSheet","displayName")
                               for e in (g(n,rel,"edges",default=[]) or [])
                               if g(e,"node","factSheet","displayName")]
def dedupe(seq):
    out=[]
    for x in seq:
        if x and x not in out: out.append(x)
    return out
def rels(*rl): return lambda n: ", ".join(dedupe(sum((rel_names(n,r) for r in rl), []))) or None
def n_users(n):
    tot, found = 0, False
    for e in (g(n,"relApplicationToOrganization","edges",default=[]) or []):
        nu = g(e,"node","numberOfUsers")
        if isinstance(nu,(int,float)): tot += nu; found = True
    return str(int(tot)) if found else None
def subs(keys):
    def f(n):
        out=[]
        for e in (g(n,"subscriptions","edges",default=[]) or []):
            names=[(r.get("name") or "") for r in (g(e,"node","roles",default=[]) or [])]
            if any(kw.lower() in rn.lower() for rn in names for kw in keys):
                u=g(e,"node","user",default={}) or {}
                lbl=" ".join(x for x in [u.get("firstName"),u.get("lastName")] if x).strip()
                em=u.get("email")
                out.append(f"{lbl} <{em}>" if (lbl and em) else (em or lbl))
        return "; ".join(dedupe([x for x in out if x])) or None
    return f
def lifecycle(n): return g(n,"lifecycle","asString") or None
def ciat(n):
    vals=[fmt(n.get(k)) for k in ["CIATConfidentiality","CIATIntegrity","CIATAvailability","CIATTracability"]]
    return " / ".join(v for v in vals if v) or None
def ai(n):
    h=[t.get("name") for t in (n.get("tags") or [])
       if "ai" in (t.get("name") or "").lower() or "ai" in (g(t,"tagGroup","name") or "").lower()]
    return ", ".join(dedupe(h)) or None
NA = lambda n: None

SECTIONS = {
 "IDENTITY & GOVERNANCE": [
   ("Name", field("displayName")), ("Aliases", field("alias")),
   ("Functional description", field("description")), ("LeanIX ID", field("id")),
   ("Lifecycle", lifecycle), ("BU / GBU", rels("relApplicationToOrganization")),
   ("Business Owner", subs(ROLE_BUSINESS_OWNER)), ("Product / Service Owner", subs(ROLE_IT_OWNER)),
   ("Technical / Run Owner", subs(ROLE_TECHNICAL)), ("DPO contact", subs(ROLE_DPO)),
   ("Cybersecurity referent", subs(ROLE_CISO)),
   ("Vendor / Integrator", rels("relApplicationToProvider","relApplicationToITComponent")),
   ("Vendor cybersecurity SPoC", NA),
   ("Vendor Cloud Service usage", combo("lxHostingType","CloudServiceProvider")),
 ],
 "BUSINESS CONTEXT & CRITICALITY": [
   ("Supported business process", rels("relApplicationToBusinessCapability")),
   ("Business criticality", field("businessCriticality")), ("CIAT", ciat),
   ("CIAT - Criticality", NA), ("Privacy score", field("PrivacySensitivityScore")),
   ("RTO", field("RTO")), ("RPO", field("RPO")), ("QoS", field("QoS")),
 ],
 "ARCHITECTURE & HOSTING": [
   ("Application type", field("ApplicationType")), ("Application scope", field("AssetType")),
   ("Tenancy", NA), ("Environments", field("ApplicationEnvironments")),
   ("Hosting type", combo("lxHostingType","CloudServiceProvider","OtherCloudServiceProvider")),
   ("Regions of hosting", combo("Region","HostingArea")),
   ("Internet facing", field("ApplicationExposure")),
 ],
 "DATA & PRIVACY": [
   ("Data types", combo("TypeOfPersonalData","PersonalDataProcessed")),
   ("Internal classification", NA), ("Number of users", n_users),
   ("Record of processing activities (GDPR)", NA),
   ("DPIA", combo("PrivacyAssessmentDone","DateOfPrivacyAssessment")),
   ("Information notice", NA), ("Cookie banner", NA),
   ("Public website without authentication", NA), ("AI usage", ai),
   ("List of service providers", rels("relApplicationToProvider","relApplicationToITComponent")),
   ("List of dependencies", rels("relToRequires","relToRequiredBy","relProviderApplicationToInterface","relConsumerApplicationToInterface")),
   ("Encryption in transit", NA), ("Encryption at rest", NA),
 ],
 "ACCESS & IAM": [
   ("User populations", NA), ("Authentication", combo("AuthenticationMethod","OtherAuthenticationMethod")),
   ("MFA", field("MFALevel")), ("Identity source", NA),
   ("Okta MFA configuration", field("MFALevel")), ("Access review frequency", NA),
 ],
 "SECURITY & MONITORING": [
   ("GSOC SIEM integration", NA), ("Patching frequency", NA),
   ("Vulnerability scans frequency", field("AutomatedSecurityScans")),
   ("Last pentest date", field("DateOfLastPenetrationTest")),
   ("Pentest Remediation Plan status", NA),
 ],
 "RISK & COMPLIANCE": [
   ("Risk assessment", combo("ISPDone","DateOfEvaluation")),
   ("Security Audit", combo("PenetrationTestDone","DateOfLastPenetrationTest")),
   ("Cybersecurity clauses in contracts", NA), ("SAP", combo("SAPDone","SAPDate")),
   ("AI Act assessment", NA), ("DRP documentation", combo("DRPObligation","DRPAvailability")),
   ("DRP test", combo("DRPLastTestDate","DRPNextTestDate")),
   ("Applicable standards identified", NA),
 ],
}

nodes = [e["node"] for e in result["data"]["allFactSheets"]["edges"] if "node" in e]

print(f"→ Transforming {len(nodes)} applications into dimensional model...")

tables = {}
for section, cols in SECTIONS.items():
    rows = []
    for n in nodes:
        row = {"Application": n.get("displayName") or n.get("name") or n.get("id"), "LeanIX ID": n.get("id")}
        for label, fn in cols:
            try: row[label] = fn(n)
            except Exception: row[label] = None
        rows.append(row)
    tables[section] = pd.DataFrame(rows)

def snake(s): return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()

KEEP_AS_TEXT = {
    "Functional description", "Aliases",
    "Business Owner", "Product / Service Owner", "Technical / Run Owner",
    "DPO contact", "Cybersecurity referent",
    "Vendor / Integrator", "List of service providers", "List of dependencies",
    "Supported business process", "BU / GBU", "Number of users","Security Audit","SAP","Risk assessment","Last pentest date","DRP test","DPIA",
}
DROP_COLS = {"Application", "LeanIX ID", "Name"}

seen, app_rows, app_id_of = set(), [], {}
for n in nodes:
    lid = n.get("id")
    if lid in seen:
        continue
    seen.add(lid)
    sid = len(app_rows) + 1
    app_id_of[lid] = sid
    app_rows.append({"application_id": sid, "leanix_id": lid, "application_name": n.get("displayName") or n.get("name") or n.get("id")})
dim_application = pd.DataFrame(app_rows)

dims = {"application": dim_application}
facts = {}

for section, df in tables.items():
    out = pd.DataFrame()
    out["application_id"] = df["LeanIX ID"].map(app_id_of)
    for col in df.columns:
        if col in DROP_COLS:
            continue
        serie = df[col]
        if serie.isna().all():
            continue
        if col in KEEP_AS_TEXT:
            out[snake(col)] = list(serie)
            continue
        k = snake(col)
        distinct = sorted({v for v in serie if pd.notna(v)}, key=str)
        id_of = {v: i for i, v in enumerate(distinct, start=1)}
        dims[k] = pd.DataFrame(
            [{f"{k}_id": 0, f"{k}_value": "(vide)"}]
            + [{f"{k}_id": i, f"{k}_value": v} for v, i in id_of.items()]
        )
        out[f"{k}_id"] = [id_of.get(v, 0) if pd.notna(v) else 0 for v in serie]
    facts["fact_" + snake(section)] = out

print(f"✓ Created {len(facts)} fact tables and {len(dims)} dimension tables")

# ============================================================================
# WRITE TO DATABRICKS
# ============================================================================

print(f"→ Writing tables to {catalog}.{schema}...")

def write_table(name, pdf):
    (spark.createDataFrame(pdf)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.{name}"))
    print(f"  ✓ {catalog}.{schema}.{name}  ({pdf.shape[0]} rows × {pdf.shape[1]} cols)")

for name, pdf in dims.items():
    write_table(f"dim_{name}", pdf)
for name, pdf in facts.items():
    write_table(name, pdf)

# ============================================================================
# EXPORT TO S3 CDH
# ============================================================================

print(f"→ Exporting tables to S3 CDH: {cdh_bucket}...")

all_tables = [f"dim_{name}" for name in dims.keys()] + list(facts.keys())

for table_name in all_tables:
    df = spark.table(f"{catalog}.{schema}.{table_name}")
    output_path = f"{cdh_bucket}{table_name}"

    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("delta.enableDeletionVectors", "false") \
        .option("overwriteSchema", "true") \
        .save(output_path)

    row_count = df.count()
    print(f"  ✓ {output_path}  ({row_count} rows)")

print(f"\n✅ Pipeline completed successfully!")
print(f"   Environment: {environment}")
print(f"   Timestamp: {datetime.now().isoformat()}")

In [ ]:
import os
import requests
import json
import pandas as pd
import re

LEANIX_API_TOKEN = os.environ.get(
    'LEANIX_API_TOKEN',
    'LXT_nEC7P6kKhJumJKhr98fgvNZwGCagCHRS6j7YAHAy'
)
LEANIX_SUBDOMAIN = 'portfolio'
LEANIX_GRAPHQL_URL = f'https://{LEANIX_SUBDOMAIN}.leanix.net/services/pathfinder/v1/graphql'
LEANIX_OAUTH2_URL = f'https://{LEANIX_SUBDOMAIN}.leanix.net/services/mtm/v1/oauth2/token'


def _obtain_access_token() -> str:
    """Obtains a LeanIX Access token using the Technical User generated
    API secret.

    Returns:
        str: The LeanIX Access Token
    """
    if not LEANIX_API_TOKEN:
        raise Exception('A valid token is required')
    response = requests.post(
        LEANIX_OAUTH2_URL,
        auth=("apitoken", LEANIX_API_TOKEN),
        data={"grant_type": "client_credentials"},
    )
    response.raise_for_status()
    return response.json().get('access_token')


access_token = _obtain_access_token()

# 1) On DECLARE les variables dans l'opération : query (...)
#    - $filter : FilterInput
#    - $sortings : [Sorting]
graphql_query = """query allFactSheetsQuery($filter: FilterInput!, $sortings: [Sorting]) {    
  allFactSheets(first: 816, filter: $filter, sort: $sortings) {
    totalCount
    pageInfo {
      hasNextPage
      hasPreviousPage
      startCursor
      endCursor
    }
    edges {
      node {
        ... on Application {
          displayName
          name
          description
          category
          completion {
            percentage
          }
          id
          fullName
          type
          status
          level
          createdAt
          updatedAt
          lxState
          alias
          externalId {
            externalId
          }
          LoginURL
          QoS
          lxCatalogStatus
          lifecycle {
            asString
            phases {
              phase
              startDate
              milestoneId
            }
          }
          CAPEX_per_year
          OPEX_per_year
          CAPEX2024
          OPEX2024
          CAPEX2025
          OPEX2025
          CAPEX2026
          OPEX2026
          CurrencyFinancail
          CostCenter
          OPEXReference
          lxTimeClassification
          lxTimeClassificationDescription
          businessCriticality
          functionalSuitability
          businessCriticalityDescription
          functionalSuitabilityDescription
          UserSupportHours
          ApplicationType
          AuthenticationMethod
          OtherAuthenticationMethod
          AssetType
          ApplicationExposure
          ZPAusage
          ZPArestriction
          PartnerAccessZscaler
          AssetPriority
          AdditionalSecurityMeasures
          AutomatedSecurityScans
          lxHostingType
          HostingArea
          CloudServiceProvider
          OtherCloudServiceProvider
          CloudAccount
          Region
          location {
            asString
          }
          ApplicationEnvironments
          ISPDone
          DateOfEvaluation
          CIATConfidentiality
          CIATIntegrity
          CIATAvailability
          CIATTracability
          SAPDone
          SAPDate
          PenetrationTestDone
          DateOfLastPenetrationTest
          PrivacyAssessmentDone
          DateOfPrivacyAssessment
          ProcomValidation
          PersonalDataProcessed
          TypeOfPersonalData
          PrivacySensitivityScore
          PrivacyComplianceLevel
          DRPObligation
          DRPAvailability
          RTO
          RPO
          DRPFearedEvents
          DRPLastTestDate
          DRPNextTestDate
          BackupStrategy
          BackupImplemented
          BackupDeletionProtection
          AccessibilityAuditDone
          LastAccessibilityAuditDate
          DigitalAccessibiltyScore
          DigitalAccessibilityStatement
          DateDigitalAccessibilityStatement
          relApplicationToApplicationPortfolio {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToPredecessor {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToSuccessor {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToStandardSolution {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToRequires {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToRequiredBy {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToParent {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relToChild {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relInstantiateApplicationToInstantiatedApplication {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relInstantiatedApplicationToInstantiateApplication {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToInitiative {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToBusinessCapability {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToBusinessContext {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToOrganization {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToDataObject {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relProviderApplicationToInterface {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relConsumerApplicationToInterface {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToProvider {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          relApplicationToITComponent {
            edges {
              node {
                id
                factSheet {
                  displayName
                  id
                  type
                }
              }
            }
          }
          subscriptions {
            edges {
              node {
                id
                user {
                  id
                  firstName
                  lastName
                  email
                  permission {
                    role
                    status
                  }
                }
                type
                roles {
                  id
                  name
                  subscriptionType
                  comment
                }
                createdAt
              }
            }
          }
          tags {
            id
            name
            color
            tagGroup {
              name
              shortName
            }
          }
          rev
          naFields
          permissions {
            create
            read
            update
            delete
          }
        }
      }
    }
  }
}

"""

#variables pour avoir r&f
variables = {
    "filter": {
        "facetFilters": [
            {"facetKey": "FactSheetTypes", "operator": "OR", "keys": ["Application"]},
            {
                "facetKey": "00000000-0000-0000-0000-000000000000",
                "operator": "OR",
                "keys": ["00000000-0000-0000-0000-000000000000"],
            },
        ]
    },
    "sortings": [{"key": "displayName", "order": "asc"}],
}

data = {'query': graphql_query, 'variables': variables}
auth_header = f'Bearer {access_token}'

response = requests.post(
    url=LEANIX_GRAPHQL_URL,
    headers={'Authorization': auth_header},
    data=json.dumps(data),
)
response.raise_for_status()
#print(json.dumps(response.json(), indent=2, ensure_ascii=False))

result = response.json()  

output_path = "/Volumes/labanalysis_dev/default/smartchunking_test/leanix_result.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

# Contrôle : doit afficher 40 (pas tronqué)
edges = result.get("data", {}).get("allFactSheets", {}).get("edges", [])
print("Fichier écrit :", output_path)
print("Nombre d'applications :", len(edges))


nodes = [e["node"] for e in result["data"]["allFactSheets"]["edges"] if "node" in e]
print(len(nodes), "applications")

# --- libellés réels des rôles de subscription (portfolio) ---
ROLE_BUSINESS_OWNER = ["Business Application Owner"]   # inclut "(Back-up)"
ROLE_IT_OWNER       = ["IT Application Owner"]
ROLE_TECHNICAL      = ["Architect", "IT Contact"]      # pas de rôle "Technical" -> archi/contact IT
ROLE_DPO            = ["Data Privacy Manager"]
ROLE_CISO           = ["CISO"]                          # inclut "Local CISO"

# --- helpers ---
def g(d, *p, default=None):
    cur = d
    for k in p:
        if not isinstance(cur, dict): return default
        cur = cur.get(k)
        if cur is None: return default
    return cur
def fmt(v):
    if v is None: return None
    if isinstance(v, bool): return "Oui" if v else "Non"
    if isinstance(v, list): return ", ".join(str(x) for x in v if x not in (None, "")) or None
    return str(v).strip() or None
def field(k): return lambda n: fmt(n.get(k))
def combo(*ks, sep=" — "): return lambda n: sep.join(x for x in (fmt(n.get(k)) for k in ks) if x) or None
def rel_names(n, rel): return [g(e,"node","factSheet","displayName")
                               for e in (g(n,rel,"edges",default=[]) or [])
                               if g(e,"node","factSheet","displayName")]
def dedupe(seq):
    out=[]
    for x in seq:
        if x and x not in out: out.append(x)
    return out
def rels(*rl): return lambda n: ", ".join(dedupe(sum((rel_names(n,r) for r in rl), []))) or None
def n_users(n):
    tot, found = 0, False
    for e in (g(n,"relApplicationToOrganization","edges",default=[]) or []):
        nu = g(e,"node","numberOfUsers")
        if isinstance(nu,(int,float)): tot += nu; found = True
    return str(int(tot)) if found else None
def subs(keys):
    def f(n):
        out=[]
        for e in (g(n,"subscriptions","edges",default=[]) or []):
            names=[(r.get("name") or "") for r in (g(e,"node","roles",default=[]) or [])]
            if any(kw.lower() in rn.lower() for rn in names for kw in keys):
                u=g(e,"node","user",default={}) or {}
                lbl=" ".join(x for x in [u.get("firstName"),u.get("lastName")] if x).strip()
                em=u.get("email")
                out.append(f"{lbl} <{em}>" if (lbl and em) else (em or lbl))
        return "; ".join(dedupe([x for x in out if x])) or None
    return f
def lifecycle(n): return g(n,"lifecycle","asString") or None
def ciat(n):
    vals=[fmt(n.get(k)) for k in ["CIATConfidentiality","CIATIntegrity","CIATAvailability","CIATTracability"]]
    return " / ".join(v for v in vals if v) or None
def ai(n):
    h=[t.get("name") for t in (n.get("tags") or [])
       if "ai" in (t.get("name") or "").lower() or "ai" in (g(t,"tagGroup","name") or "").lower()]
    return ", ".join(dedupe(h)) or None
NA = lambda n: None

# --- 7 sections : (libellé de colonne, extracteur) ---
SECTIONS = {
 "IDENTITY & GOVERNANCE": [
   ("Name", field("displayName")), ("Aliases", field("alias")),
   ("Functional description", field("description")), ("LeanIX ID", field("id")),
   ("Lifecycle", lifecycle), ("BU / GBU", rels("relApplicationToOrganization")),
   ("Business Owner", subs(ROLE_BUSINESS_OWNER)), ("Product / Service Owner", subs(ROLE_IT_OWNER)),
   ("Technical / Run Owner", subs(ROLE_TECHNICAL)), ("DPO contact", subs(ROLE_DPO)),
   ("Cybersecurity referent", subs(ROLE_CISO)),
   ("Vendor / Integrator", rels("relApplicationToProvider","relApplicationToITComponent")),
   ("Vendor cybersecurity SPoC", NA),
   ("Vendor Cloud Service usage", combo("lxHostingType","CloudServiceProvider")),
 ],
 "BUSINESS CONTEXT & CRITICALITY": [
   ("Supported business process", rels("relApplicationToBusinessCapability")),
   ("Business criticality", field("businessCriticality")), ("CIAT", ciat),
   ("CIAT - Criticality", NA), ("Privacy score", field("PrivacySensitivityScore")),
   ("RTO", field("RTO")), ("RPO", field("RPO")), ("QoS", field("QoS")),
 ],
 "ARCHITECTURE & HOSTING": [
   ("Application type", field("ApplicationType")), ("Application scope", field("AssetType")),
   ("Tenancy", NA), ("Environments", field("ApplicationEnvironments")),
   ("Hosting type", combo("lxHostingType","CloudServiceProvider","OtherCloudServiceProvider")),
   ("Regions of hosting", combo("Region","HostingArea")),
   ("Internet facing", field("ApplicationExposure")),
 ],
 "DATA & PRIVACY": [
   ("Data types", combo("TypeOfPersonalData","PersonalDataProcessed")),
   ("Internal classification", NA), ("Number of users", n_users),
   ("Record of processing activities (GDPR)", NA),
   ("DPIA", combo("PrivacyAssessmentDone","DateOfPrivacyAssessment")),
   ("Information notice", NA), ("Cookie banner", NA),
   ("Public website without authentication", NA), ("AI usage", ai),
   ("List of service providers", rels("relApplicationToProvider","relApplicationToITComponent")),
   ("List of dependencies", rels("relToRequires","relToRequiredBy","relProviderApplicationToInterface","relConsumerApplicationToInterface")),
   ("Encryption in transit", NA), ("Encryption at rest", NA),
 ],
 "ACCESS & IAM": [
   ("User populations", NA), ("Authentication", combo("AuthenticationMethod","OtherAuthenticationMethod")),
   ("MFA", field("MFALevel")), ("Identity source", NA),
   ("Okta MFA configuration", field("MFALevel")), ("Access review frequency", NA),
 ],
 "SECURITY & MONITORING": [
   ("GSOC SIEM integration", NA), ("Patching frequency", NA),
   ("Vulnerability scans frequency", field("AutomatedSecurityScans")),
   ("Last pentest date", field("DateOfLastPenetrationTest")),
   ("Pentest Remediation Plan status", NA),
 ],
 "RISK & COMPLIANCE": [
   ("Risk assessment", combo("ISPDone","DateOfEvaluation")),
   ("Security Audit", combo("PenetrationTestDone","DateOfLastPenetrationTest")),
   ("Cybersecurity clauses in contracts", NA), ("SAP", combo("SAPDone","SAPDate")),
   ("AI Act assessment", NA), ("DRP documentation", combo("DRPObligation","DRPAvailability")),
   ("DRP test", combo("DRPLastTestDate","DRPNextTestDate")),
   ("Applicable standards identified", NA),
 ],
}

# --- construction : 1 DataFrame par section (1 ligne / application) ---
def app_id(n): return n.get("displayName") or n.get("name") or n.get("id")
tables = {}
for section, cols in SECTIONS.items():
    rows = []
    for n in nodes:
        row = {"Application": app_id(n), "LeanIX ID": n.get("id")}
        for label, fn in cols:
            try: row[label] = fn(n)
            except Exception: row[label] = None
        rows.append(row)
    tables[section] = pd.DataFrame(rows)



def snake(s): return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()

# Colonnes laissées en TEXTE dans les faits (haute cardinalité, pas de sens en dimension).
# Pour transformer l'une d'elles en dimension, retire-la simplement de ce set.
KEEP_AS_TEXT = {
    "Functional description", "Aliases",
    "Business Owner", "Product / Service Owner", "Technical / Run Owner",
    "DPO contact", "Cybersecurity referent",
    "Vendor / Integrator", "List of service providers", "List of dependencies",
    "Supported business process", "BU / GBU", "Number of users","Security Audit","SAP","Risk assessment","Last pentest date","DRP test","DPIA",
}
DROP_COLS = {"Application", "LeanIX ID", "Name"}   # redondant avec dim_application

# 1) Dimension partagée : application (la clé qui relie les 7 faits)
seen, app_rows, app_id_of = set(), [], {}
for n in nodes:
    lid = n.get("id")
    if lid in seen:
        continue
    seen.add(lid)
    sid = len(app_rows) + 1
    app_id_of[lid] = sid
    app_rows.append({"application_id": sid, "leanix_id": lid, "application_name": app_id(n)})
dim_application = pd.DataFrame(app_rows)

# 2) Construction des faits + dimensions
dims = {"application": dim_application}
facts = {}
for section, df in tables.items():
    out = pd.DataFrame()
    out["application_id"] = df["LeanIX ID"].map(app_id_of)
    for col in df.columns:
        if col in DROP_COLS:
            continue
        serie = df[col]
        if serie.isna().all():          # colonne vide / hors LeanIX -> ignorée
            continue
        if col in KEEP_AS_TEXT:
            out[snake(col)] = list(serie)
            continue
        # dimension : valeurs distinctes -> id incrémental (0 = (vide))
        k = snake(col)
        distinct = sorted({v for v in serie if pd.notna(v)}, key=str)
        id_of = {v: i for i, v in enumerate(distinct, start=1)}
        dims[k] = pd.DataFrame(
            [{f"{k}_id": 0, f"{k}_value": "(vide)"}]
            + [{f"{k}_id": i, f"{k}_value": v} for v, i in id_of.items()]
        )
        out[f"{k}_id"] = [id_of.get(v, 0) if pd.notna(v) else 0 for v in serie]
    facts["fact_" + snake(section)] = out

# --- Aperçu avant écriture : faits puis dimensions ---
print(f"{len(facts)} tables de faits | {len(dims)} tables de dimension\n")

print("################  TABLES DE FAITS  ################")
for name, pdf in facts.items():
    print(f"\n========== {name}  ({pdf.shape[0]} lignes x {pdf.shape[1]} colonnes) ==========")
    display(pdf)

print("\n################  TABLES DE DIMENSION  ################")
for name, pdf in dims.items():
    print(f"\n========== dim_{name}  ({pdf.shape[0]} lignes) ==========")
    display(pdf)


In [ ]:
def write_table(name, pdf):
    (spark.createDataFrame(pdf)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"leanix_dev.default.{name}"))
    print(f"écrit : leanix_dev.default.{name}  ({pdf.shape[0]}x{pdf.shape[1]})")

for name, pdf in dims.items():
    write_table(f"dim_{name}", pdf)
for name, pdf in facts.items():
    write_table(name, pdf)

In [ ]:

catalog = "leanix_dev"
schema  = "default"
cdh_bucket = "s3://demo-data-bucket/"

# Liste des tables à exporter vers CDH
dim_names  = [f"dim_{name}" for name in dims.keys()]
fact_names = list(facts.keys())
all_tables = dim_names + fact_names

for table_name in all_tables:
    df = spark.table(f"{catalog}.{schema}.{table_name}")
    output_path = f"{cdh_bucket}{table_name}"

    print(output_path)

    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("delta.enableDeletionVectors", "false") \
        .option("overwriteSchema", "true") \
        .save(output_path)

    print(f"écrit sur CDH : {output_path}  ({df.count()} lignes)")